# Alignment Transfer：standalone実行例

T2TT → Attention・CTC・HuBERT → alignment診断 → 5条件のS2UT学習 → 推論・評価を順に確認する旧trainerの実行例です。

> 学習済みbenchmark Baselineとの本比較・ZIPによるDrive保存と復元には[benchmark比較ノートブック](https://colab.research.google.com/github/Yaaamashiro/alignment-transfer-s2ut/blob/main/notebooks/colab/benchmark_comparison.ipynb)を使用します。このノートブックは`--trainer legacy`とDriveへの直接保存を使います。

GPUランタイムで上から実行します。既定の`UPDATES=2`は接続確認用です。実験条件を変更する場合は新しいROOTを使います。

## 1. Drive接続と設定

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
REPO_URL = 'https://github.com/Yaaamashiro/alignment-transfer-s2ut.git' #@param {type:"string"}
REPO_REVISION = '11abfcb0a9709bd122c7687a35975e08a122c402' #@param {type:"string"}
ROOT = '/content/drive/MyDrive/alignment-transfer-s2ut-data/experiment-stream-001' #@param {type:"string"}
CORPUS = '/content/drive/MyDrive/ja-en-direct-s2st-corpus-data' #@param {type:"string"}
RECIPE = 'controlled' #@param ['controlled', 'strong']
RUNS = ROOT + '/' + RECIPE
ANNEAL_STEPS = 10000 #@param {type:"integer"}
UPDATES = 2 #@param {type:"integer"}
PYTHON = '/content/alignment-venv/bin/python'
os.environ['HF_HOME'] = '/content/huggingface'

## 2. 環境の構築と検査

In [ ]:
# Clone/install only. Never point outputs at the corpus or benchmark repositories.
import subprocess, sys

def execute(*arguments):
    return subprocess.run(list(map(str, arguments)), check=True)
from pathlib import Path
assert len(REPO_REVISION) == 40 and 'SET_' not in REPO_URL
assert not Path(ROOT).resolve().is_relative_to(Path(CORPUS).resolve())
if not Path('/content/alignment-transfer-s2ut/.git').exists():
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, '/content/alignment-transfer-s2ut'], check=True)
subprocess.run(['git', '-C', '/content/alignment-transfer-s2ut', 'fetch', 'origin', REPO_REVISION], check=True)
subprocess.run(['git', '-C', '/content/alignment-transfer-s2ut', 'checkout', '--detach', REPO_REVISION], check=True)
os.chdir('/content/alignment-transfer-s2ut')
execute(
    sys.executable, '-m', 'pip', '-q', 'install', 'uv'
)
execute(
    'uv', 'venv', '--python', '3.10.22', '--seed', '--allow-existing', '/content/alignment-venv'
)
execute(
    PYTHON, '-m', 'pip', '-q', 'install', 'pip==23.3.2', 'setuptools<81', 'wheel'
)
execute(
    PYTHON, '-m', 'pip', '-q', 'install', 'torch==2.7.1', 'torchaudio==2.7.1', 'numpy==1.26.4',
    'Cython==3.2.9'
)
execute(
    PYTHON, '-m', 'pip', 'install', '--no-build-isolation', '-r', 'requirements-fairseq.txt'
)
execute(
    PYTHON, '-m', 'pip', 'install', '-r', 'requirements.txt'
)
execute(
    PYTHON, '-m', 'pytest', '-q'
)
execute(PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'runtime')
execute(PYTHON, '-m', 'pip', 'check')

## 3. Manifestと共通構成の確認

In [ ]:
# First import only. On resume, retain the existing canonical manifest unchanged.
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.prepare_manifest', '--input',
    f'{CORPUS}/production/manifests/releases/accepted.jsonl', '--corpus-root', f'{CORPUS}',
    '--output', f'{ROOT}/manifest.jsonl', '--resume'
)
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.compare_architectures', '--t2tt-config',
    'configs/t2tt/fisher.yaml', '--s2ut-config', 'configs/s2ut/fisher.yaml', '--output',
    f'{ROOT}/architecture.json', '--resume'
)

## 4. T2TT学習

In [ ]:
# Keep the same recipe; resume automatically if a checkpoint exists.
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.train_t2tt', '--manifest', f'{ROOT}/manifest.jsonl',
    '--config', 'configs/t2tt/fisher.yaml', '--output', f'{ROOT}/t2tt', '--max-updates',
    f'{UPDATES}', '--device', 'cuda', '--resume-if-exists'
)

## 5. Attention・CTC・unitとteacherの作成

In [ ]:
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.extract_t2tt_attention', '--manifest',
    f'{ROOT}/manifest.jsonl', '--checkpoint', f'{ROOT}/t2tt/checkpoint_best.pt', '--output',
    f'{ROOT}/mt', '--device', 'cuda', '--resume'
)
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.align_speech_text', '--manifest',
    f'{ROOT}/manifest.jsonl', '--config', 'configs/alignment/ctc.yaml', '--output', f'{ROOT}/ctc',
    '--device', 'cuda', '--resume'
)
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.fetch_assets', '--config',
    'configs/units/hubert_km100.yaml'
)
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.extract_units', '--manifest', f'{ROOT}/manifest.jsonl',
    '--config', 'configs/units/hubert_km100.yaml', '--output', f'{ROOT}/units', '--device', 'cuda',
    '--resume'
)
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.build_teacher_alignment', '--manifest',
    f'{ROOT}/manifest.jsonl', '--mt-attention', f'{ROOT}/mt', '--alignment', f'{ROOT}/ctc',
    '--units', f'{ROOT}/units', '--config', 'configs/s2ut/fisher.yaml', '--output',
    f'{ROOT}/teacher', '--resume'
)

## 6. Alignment診断

In [ ]:
# Audit coverage/uncertainty before training; inspect heatmaps against original audio.
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.diagnose_alignment', '--manifest',
    f'{ROOT}/manifest.jsonl', '--teacher', f'{ROOT}/teacher', '--units', f'{ROOT}/units',
    '--config', 'configs/s2ut/fisher.yaml', '--split', 'dev', '--max-samples', '500', '--heatmaps',
    '20', '--source-mapping', 'frontend', '--prior-smoothing', '0.05', '--output',
    f'{ROOT}/alignment-audit', '--overwrite'
)

`RECIPE`でcontrolled / strongを選択します。strongは文字補助decoder・decoder CTC・CMVN・SpecAugmentを使用し、両recipeとも単調prior対照を含めて比較します。出力先はrecipe別です。UPDATES=2は接続確認、pilotは5000〜20000更新、本学習は全条件共通の更新数を設定してください。過去の実装からのresumeはcontractの不一致で停止するため、新しいROOTを使用します。

## 7. 5条件のS2UT学習

In [ ]:
# One recipe, timing and smoothing across all five conditions.
SHARED = ['--trainer', 'legacy', '--manifest', f'{ROOT}/manifest.jsonl',
    '--config', 'configs/s2ut/fisher.yaml', '--units', f'{ROOT}/units',
    '--loader-workers', 2, '--prefetch-batches', 2, '--cache-dir', '/content/alignment-transfer-cache',
    '--cache-gb', 8, '--pin-memory', '--recipe', RECIPE, '--prior-smoothing', 0.05,
    '--source-mapping', 'frontend', '--anneal-steps', ANNEAL_STEPS,
    '--validation-interval', 1000, '--save-interval-updates', 1000,
    '--max-updates', UPDATES, '--device', 'cuda', '--resume-if-exists']
CONDITIONS = [('baseline', 'baseline', 't2tt'), ('distillation', 'distillation', 't2tt'),
    ('guidance', 'guidance', 't2tt'), ('annealed_guidance', 'annealed_guidance', 't2tt'),
    ('annealed_monotonic', 'annealed_guidance', 'monotonic')]
for name, method, prior in CONDITIONS:
    teacher = ['--teacher', f'{ROOT}/teacher'] if name not in ('baseline', 'annealed_monotonic') else []
    control = ['--prior-type', 'monotonic'] if prior == 'monotonic' else []
    execute(PYTHON, '-m', 'alignment_transfer_s2ut.train_s2ut', *SHARED,
        '--method', method, '--output', f'{RUNS}/{name}', *teacher, *control)
execute(PYTHON, '-m', 'alignment_transfer_s2ut.compare_experiments',
    *[f'{RUNS}/{name}' for name, _, _ in CONDITIONS])


推論入力には`pair_id, ja_audio`だけを持つJSONLを用意します。英語referenceやtranscriptを含む学習manifestは渡しません。評価に使うsplitとID集合を一致させてください。

## 8. 推論と最終評価

**最終評価用です。** 学習条件の調整にはdevを使い、testを選択基準にしません。2更新の接続確認結果は研究上のスコアとして扱いません。

In [ ]:
SPEECH_ONLY = ROOT + '/test_speech_only.jsonl' #@param {type:"string"}
METHOD = 'annealed_guidance' #@param ['baseline', 'distillation', 'guidance', 'annealed_guidance', 'annealed_monotonic']
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.prepare_inference_input', '--manifest',
    f'{ROOT}/manifest.jsonl', '--split', 'test', '--output', f'{SPEECH_ONLY}', '--resume'
)
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.fetch_assets', '--config', 'configs/s2ut/vocoder.yaml'
)
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.infer', '--input', f'{SPEECH_ONLY}', '--checkpoint',
    f'{RUNS}/{METHOD}/checkpoint_best.pt', '--vocoder-config', 'configs/s2ut/vocoder.yaml',
    '--output', f'{RUNS}/{METHOD}/test-predictions.jsonl', '--device', 'cuda', '--resume'
)
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.evaluate', '--manifest', f'{ROOT}/manifest.jsonl',
    '--split', 'test', '--predictions', f'{RUNS}/{METHOD}/test-predictions.jsonl', '--units',
    f'{ROOT}/units', '--asr-config', 'configs/experiments/evaluation.yaml', '--output',
    f'{RUNS}/{METHOD}/test-metrics.json', '--device', 'cuda', '--resume'
)

## 保存結果と再開

`ROOT/t2tt/`にteacher、`ROOT/teacher/`に投影済みalignment、`ROOT/<recipe>/<condition>/`に各学習結果を保存します。各CLIは失敗時に停止します。同じ設定・ROOTで再実行すると、完成済みartifactとcheckpointを再利用します。

この旧経路はcheckpointをDriveへ直接保存します。本比較ではbenchmark比較ノートブックのローカル作業・ZIP保存を使用してください。